# 2. Export and interchange

One model, four serializations:

| Format | Functions | Notes |
|---|---|---|
| SysML v2 text | `to_sysml` | re-parseable; round-trips preserve the model |
| JSON | `to_json` / `from_json` | lossless; also the cache format (no pickles) |
| KerML | `to_kerml` | one-way projection onto the kernel language |
| OMG API JSON | `longeron.api` | flat `@type`/`@id` records (needs pyecore) |

**You will learn how to:**

- round-trip a model through JSON and keep executing it;
- save and load models by file suffix (`save` / `load`);
- project a model onto KerML (`to_kerml`);
- project a model onto the OMG spec metamodel and API JSON.

**Prerequisites:** tutorial 1 (parsing and the object model). The last
section needs the `ecore` extra and skips itself when pyecore is
missing.

The cell below parses the demo model and prints its regenerated SysML
text.

In [ ]:
import longeron

model = longeron.loads("""
package Demo {
    part def Battery { attribute capacity : Real = 5200.0; }
    part def Drone {
        attribute mass : Real = 1.2;
        part battery : Battery;
    }
    calc def HoverTime { in c : Real; return : Real = c / 12000.0 * 60.0; }
}
""")
print(longeron.to_sysml(model))

## JSON is lossless: parse it back and keep executing

The claim is checkable, so the cell checks it: the re-imported model
equals the original, dictionary for dictionary, and a calc still runs
on the clone.

In [ ]:
json_text = longeron.to_json(model)
clone = longeron.from_json(json_text)

assert longeron.to_dict(clone) == longeron.to_dict(model)
print("round-trip preserved the model bit-for-bit")
print("HoverTime from the clone:", longeron.Interpreter(clone).call("Demo::HoverTime", c=5200.0))

## `save()` / `load()` dispatch on the file suffix

One call saves to any of the three formats. The suffix picks the
exporter.

In [ ]:
import tempfile
from pathlib import Path

out = Path(tempfile.mkdtemp())
for suffix in (".sysml", ".json", ".kerml"):
    longeron.save(model, out / f"demo{suffix}")
sorted(p.name for p in out.iterdir())

## KerML projection

SysML v2 is defined as an extension of KerML: `part def` becomes
`struct`, `calc def` becomes `function`, and constraints become `inv`.
The projection is one-way, and the output is guaranteed re-parseable by
the bundled KerML grammar. The cell checks that guarantee with
`parse_kerml_text`.

In [ ]:
kerml_text = longeron.to_kerml(model)
print(kerml_text)
longeron.parse_kerml_text(kerml_text)  # raises on invalid KerML
print("KerML output re-parses cleanly")

## OMG spec metamodel and API JSON (optional, `pip install "longeron[ecore]"`)

`longeron.ecore` projects models onto the *specification* abstract
syntax (the pilot implementation's `SysML.ecore`, 175 metaclasses), with
reified memberships and relationship elements. `longeron.api` emits the
flat records SysML v2 tools exchange. The projection report counts what
was covered and what was skipped, so nothing is silently dropped.

In [ ]:
try:
    from longeron import api, ecore
except ImportError:
    print("pyecore not installed -- skipping")
else:
    spec = ecore.to_spec(model)
    print("projection report:", spec.report)

    records = api.to_api_records(model)
    part_def = next(r for r in records if r.get("declaredName") == "Drone")
    print("\nAPI record for Drone:")
    for key, value in list(part_def.items())[:5]:
        print(f"  {key}: {value}")